# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [7]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [8]:
# 1. Crear dim_tiempo
spotify_clean = spotify_raw.copy()

# Normalizar tipos para que el ETL funcione bien
spotify_clean["Release Date"] = pd.to_datetime(spotify_clean["Release Date"], errors="coerce")
spotify_clean["Track Score"] = pd.to_numeric(spotify_clean["Track Score"], errors="coerce")

spotify_clean["Spotify Streams"] = (
    spotify_clean["Spotify Streams"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace(" ", "", regex=False)
    .replace("", np.nan)
    .astype(float)
)

spotify_clean["YouTube Views"] = (
    spotify_clean["YouTube Views"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace(" ", "", regex=False)
    .replace("", np.nan)
    .astype(float)
)

spotify_clean["Explicit Track"] = pd.to_numeric(
    spotify_clean["Explicit Track"], errors="coerce"
).fillna(0).astype(int)

# Extraer año, mes y día
spotify_clean["Year"] = spotify_clean["Release Date"].dt.year
spotify_clean["Month"] = spotify_clean["Release Date"].dt.month
spotify_clean["Day"] = spotify_clean["Release Date"].dt.day

# Dimensión tiempo sin duplicados
dim_tiempo = (
    spotify_clean[["Year", "Month", "Day"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_tiempo.insert(0, "id_tiempo", np.arange(1, len(dim_tiempo) + 1))
display(dim_tiempo.head())

,id_tiempo,Year,Month,Day
0,1,2024,4,26
1,2,2024,5,4
2,3,2024,3,19
3,4,2023,1,12
4,5,2024,5,31


In [9]:
# 2. Crear dim_pista
dim_pista = (
    spotify_clean[["Track", "Artist", "Explicit Track"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_pista.insert(0, "id_pista", np.arange(1, len(dim_pista) + 1))
display(dim_pista.head())

,id_pista,Track,Artist,Explicit Track
0,1,MILLION DOLLAR BABY,Tommy Richman,0
1,2,Not Like Us,Kendrick Lamar,1
2,3,i like the way you kiss me,Artemas,0
3,4,Flowers,Miley Cyrus,0
4,5,Houdini,Eminem,1


In [10]:
# 3. Crear fact_streams
fact_streams = spotify_clean[[
    "Track",
    "Artist",
    "Explicit Track",
    "Track Score",
    "Spotify Streams",
    "YouTube Views",
    "Year",
    "Month",
    "Day"
]].copy()

# Unir con la dimensión tiempo
fact_streams = fact_streams.merge(
    dim_tiempo,
    on=["Year", "Month", "Day"],
    how="left"
)

# Unir con la dimensión pista
fact_streams = fact_streams.merge(
    dim_pista,
    on=["Track", "Artist", "Explicit Track"],
    how="left"
)

# Mantener solo las llaves y métricas numéricas
fact_streams = fact_streams[[
    "id_tiempo",
    "id_pista",
    "Track Score",
    "Spotify Streams",
    "YouTube Views"
]].dropna(subset=["id_tiempo", "id_pista"]).reset_index(drop=True)

display(fact_streams.head())

,id_tiempo,id_pista,Track Score,Spotify Streams,YouTube Views
0,1,1,725.4,3.904709e+08,8.427475e+07
1,2,2,545.9,3.237039e+08,1.163470e+08
2,3,3,538.4,6.013093e+08,1.225991e+08
3,4,4,444.9,2.031281e+09,1.096101e+09
4,5,5,423.3,1.070349e+08,7.737396e+07


In [11]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
cubo_olap = fact_streams.merge(
    dim_tiempo,
    on="id_tiempo",
    how="left"
).merge(
    dim_pista,
    on="id_pista",
    how="left"
)

cubo_olap = cubo_olap[[
    "id_tiempo",
    "id_pista",
    "Year",
    "Month",
    "Day",
    "Track",
    "Artist",
    "Explicit Track",
    "Track Score",
    "Spotify Streams",
    "YouTube Views"
]]

display(cubo_olap.head())

,id_tiempo,id_pista,Year,Month,Day,Track,Artist,Explicit Track,Track Score,Spotify Streams,YouTube Views
0,1,1,2024,4,26,MILLION DOLLAR BABY,Tommy Richman,0,725.4,3.904709e+08,8.427475e+07
1,2,2,2024,5,4,Not Like Us,Kendrick Lamar,1,545.9,3.237039e+08,1.163470e+08
2,3,3,2024,3,19,i like the way you kiss me,Artemas,0,538.4,6.013093e+08,1.225991e+08
3,4,4,2023,1,12,Flowers,Miley Cyrus,0,444.9,2.031281e+09,1.096101e+09
4,5,5,2024,5,31,Houdini,Eminem,1,423.3,1.070349e+08,7.737396e+07


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [12]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: Roll-up
respuesta_1 = (
    cubo_olap.groupby("Artist", as_index=False)["Spotify Streams"]
    .sum()
    .sort_values("Spotify Streams", ascending=False)
)

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: Slice
respuesta_2 = cubo_olap[cubo_olap["Year"] == 2024]

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: Drill-down
respuesta_3 = (
    cubo_olap.groupby(["Year", "Month"], as_index=False)["Spotify Streams"]
    .sum()
    .sort_values(["Year", "Month"])
)

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: Dice
respuesta_4 = cubo_olap[
    (cubo_olap["Explicit Track"] == 1) &
    (cubo_olap["Year"] == 2024)
]

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: Roll-up
respuesta_5 = (
    cubo_olap.groupby("Year", as_index=False)["Track Score"]
    .mean()
    .sort_values("Year")
)

# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: Slice
respuesta_6 = cubo_olap[cubo_olap["Artist"] == "Bad Bunny"]

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: Drill-down
respuesta_7 = (
    cubo_olap.groupby(["Artist", "Track"], as_index=False)["YouTube Views"]
    .sum()
    .sort_values(["Artist", "YouTube Views"], ascending=[True, False])
)

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: Slice
respuesta_8 = cubo_olap[
    (cubo_olap["Month"] == 12) &
    (cubo_olap["Year"] == 2023)
]

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: Roll-up
respuesta_9 = (
    cubo_olap.groupby("Month", as_index=False)["Spotify Streams"]
    .sum()
    .sort_values("Spotify Streams", ascending=False)
)

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: Dice
respuesta_10 = cubo_olap[
    (cubo_olap["Artist"] == "Taylor Swift") &
    (cubo_olap["Explicit Track"] == 0) &
    (cubo_olap["Year"] == 2024)
]

In [13]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: Roll-up
respuesta_1 = (
    cubo_olap.groupby("Artist", as_index=False)["Spotify Streams"]
    .sum()
    .sort_values("Spotify Streams", ascending=False)
)

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: Slice
respuesta_2 = cubo_olap[cubo_olap["Year"] == 2024]

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: Drill-down
respuesta_3 = (
    cubo_olap.groupby(["Year", "Month"], as_index=False)["Spotify Streams"]
    .sum()
    .sort_values(["Year", "Month"])
)

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: Dice
respuesta_4 = cubo_olap[
    (cubo_olap["Explicit Track"] == 1) &
    (cubo_olap["Year"] == 2024)
]

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: Roll-up
respuesta_5 = (
    cubo_olap.groupby("Year", as_index=False)["Track Score"]
    .mean()
    .sort_values("Year")
)

# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: Slice
respuesta_6 = cubo_olap[cubo_olap["Artist"] == "Bad Bunny"]

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: Drill-down
respuesta_7 = (
    cubo_olap.groupby(["Artist", "Track"], as_index=False)["YouTube Views"]
    .sum()
    .sort_values(["Artist", "YouTube Views"], ascending=[True, False])
)

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: Slice
respuesta_8 = cubo_olap[
    (cubo_olap["Month"] == 12) &
    (cubo_olap["Year"] == 2023)
]

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: Roll-up
respuesta_9 = (
    cubo_olap.groupby("Month", as_index=False)["Spotify Streams"]
    .sum()
    .sort_values("Spotify Streams", ascending=False)
)

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: Dice
respuesta_10 = cubo_olap[
    (cubo_olap["Artist"] == "Taylor Swift") &
    (cubo_olap["Explicit Track"] == 0) &
    (cubo_olap["Year"] == 2024)
]